In [14]:
from torch import optim
from ReplayBuffer import ReplayBuffer
from DQN import DQN
from utils import (
    get_numeric_state,
    choose_action,
    apply_action,
    get_reward,
    train_step_batch,
)
import traci


sumo_binary = "sumo-gui"
sumo_config = "./scenarios/bologna/run.sumocfg"
# sumo_config = "./scenarios/SCENARIO-I/DT_GM_live24h.sumocfg"
# sumo_config = "scenarios/TAVF-Hamburg/run_simulation.sumocfg"

view_id = "View #0"

car_id = "Borgo_100_0"
control_enabled = False






In [2]:
policy_net = DQN(state_dim=7, action_dim=5)
target_net = DQN(state_dim=7, action_dim=5)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
replay_buffer = ReplayBuffer(capacity=10000)
batch_size = 32

epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05
target_update_freq = 32

control_enabled = False
prev_state = None

In [15]:
import random

traci.start([sumo_binary, "-c", sumo_config])
id_list = traci.vehicle.getIDList()

if not id_list:
    traci.simulationStep(1)
    id_list = traci.vehicle.getIDList()


car_id = id_list[random.randint(0, len(id_list) - 1)]
traci.gui.track(car_id, view_id)


id_list = traci.vehicle.getIDList()
while not id_list:
    traci.simulationStep()
    id_list = traci.vehicle.getIDList()

car_id = random.choice(id_list)
traci.gui.track(car_id, view_id)
control_enabled = False

for step in range(1000):
    # step simulation once
    traci.simulationStep()

    current_ids = traci.vehicle.getIDList()

    # if current car is gone before acting, pick another one
    if car_id not in current_ids:
        print(f"step={step}: vehicle not in sim")

        if not current_ids:
            continue

        car_id = random.choice(current_ids)
        traci.gui.track(car_id, view_id)
        control_enabled = False
        continue

    # enable external control for newly selected car
    if not control_enabled:
        traci.vehicle.setSpeedMode(car_id, 0)
        control_enabled = True

    # observe state
    state = get_numeric_state(car_id)

    # choose and apply action
    action = choose_action(policy_net, state, epsilon)
    delta_v = apply_action(car_id, action)

    # second sim step so action has effect
    traci.simulationStep()

    current_ids = traci.vehicle.getIDList()
    arrived_ids = traci.simulation.getArrivedIDList()

    # terminal handling
    if car_id not in current_ids:
        next_state = state
        done = 1.0

        if car_id in arrived_ids:
            reward = 20.0
            print(f"step={step}: vehicle arrived normally +++++++++++++++++++++++++++++++++++++++++++++++++++\n")
        else:
            reward = -50.0
            print(f"step={step}: abnormal disappearance of vehicle ------------------------------------------------ \n")

        replay_buffer.push(state, int(action), reward, next_state, done)

        loss = train_step_batch(
            policy_net=policy_net,
            target_net=target_net,
            optimizer=optimizer,
            replay_buffer=replay_buffer,
            batch_size=batch_size,
            gamma=0.99,
        )

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        if step % target_update_freq == 0:
            target_net.load_state_dict(policy_net.state_dict())

        # choose new vehicle if any remain
        current_ids = traci.vehicle.getIDList()
        if current_ids:
            car_id = random.choice(current_ids)
            traci.gui.track(car_id, view_id)
            control_enabled = False

        loss_str = f"{loss:8.5f}" if loss is not None else " warmup "
        print(
            f"step={step:03d} "
            f"car={car_id if current_ids else 'NONE'} "
            f"action={action.name:<14} "
            f"state={state} "
            f"reward={reward:6.3f} "
            f"loss={loss_str} "
            f"epsilon={epsilon:5.3f}"
        )
        continue

    # normal transition
    next_state = get_numeric_state(car_id)
    reward = get_reward(car_id, delta_v)
    done = 0.0

    replay_buffer.push(state, int(action), reward, next_state, done)

    loss = train_step_batch(
        policy_net=policy_net,
        target_net=target_net,
        optimizer=optimizer,
        replay_buffer=replay_buffer,
        batch_size=batch_size,
        gamma=0.99,
    )

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if step % target_update_freq == 0:
        target_net.load_state_dict(policy_net.state_dict())

    loss_str = f"{loss:8.5f}" if loss is not None else " warmup "

    print(
        f"----------------------------------------"
        f"step={step:03d} \n"
        f"car={car_id} \n"
        f"action={action.name:<14} \n"
        f"state={state} \n"
        f"reward={reward:6.3f} \n"
        f"loss={loss_str} \n"
        f"epsilon={epsilon:5.3f} \n"
    )

traci.close()

 Retrying in 1 seconds


Error: Could not set option 'restriction' because attribute 'value' is missing.


----------------------------------------step=000 
car=flow_car1_onramp1_N_ES.0 
action=SLOWER         
state=[1.1973247000001517, 1.0, 1.1973247000001517, 0.0, 1.0, 0.0, 0.0] 
reward= 0.997 
loss= 3.07343 
epsilon=0.050 

----------------------------------------step=001 
car=flow_car1_onramp1_N_ES.0 
action=SLOWER         
state=[1.0973247000001516, 1.0, 1.0973247000001516, 0.0, 1.0, 0.0, 0.0] 
reward= 0.897 
loss=12.45433 
epsilon=0.050 

----------------------------------------step=002 
car=flow_car1_onramp1_N_ES.0 
action=SLOWER         
state=[0.9973247000001516, 1.0, 0.9973247000001516, 0.0, 1.0, 0.0, 0.0] 
reward= 0.797 
loss= 2.40712 
epsilon=0.050 

----------------------------------------step=003 
car=flow_car1_onramp1_N_ES.0 
action=SLOWER         
state=[0.8973247000001516, 1.0, 0.8973247000001516, 0.0, 1.0, 0.0, 0.0] 
reward= 0.697 
loss=16.93157 
epsilon=0.050 

----------------------------------------step=004 
car=flow_car1_onramp1_N_ES.0 
action=SLOWER         
state=[0.

In [13]:
print(policy_net.state_dict())

OrderedDict({'net.0.weight': tensor([[ 4.9771e-01,  1.3768e-01,  2.3730e-01,  2.3271e-01, -2.1577e-01,
          1.4682e-01,  5.0576e-01],
        [ 3.8191e-02,  3.1121e-01, -4.3705e-01,  5.7821e-01,  6.1889e-01,
         -3.0896e-02, -4.4468e-02],
        [ 4.3307e-01, -3.4574e-02, -2.5398e-01,  4.9245e-01,  4.9262e-01,
         -1.3981e-01,  1.2180e-01],
        [ 1.2596e-02, -8.5964e-02,  6.2015e-02,  2.6655e-01, -7.2295e-02,
         -5.2671e-02, -6.9555e-02],
        [ 4.0261e-01,  3.7486e-01,  1.2093e-01, -3.9725e-01, -5.3097e-01,
         -2.9565e-02, -2.2953e-01],
        [ 2.5620e-01, -7.5503e-01, -4.1777e-01,  1.4775e-01,  3.7142e-01,
          1.1996e-01,  2.0868e-01],
        [ 1.9308e-01,  3.6917e-01, -4.0957e-01, -6.4734e-02,  3.2411e-01,
          1.5081e-01,  3.8274e-01],
        [ 1.4279e-01,  3.7689e-01,  3.7501e-02,  4.1496e-01,  1.0652e-01,
         -4.5365e-02, -4.4316e-01],
        [ 1.6005e-01,  4.6374e-01,  2.9385e-01, -2.7404e-01,  4.1269e-02,
         -3.1741e

In [10]:
print(policy_net.state_dict())


OrderedDict({'net.0.weight': tensor([[ 5.3260e-01,  1.4755e-01,  2.5535e-01,  2.5381e-01, -2.4357e-01,
          1.5912e-01,  5.0278e-01],
        [ 2.3108e-02,  3.3609e-01, -4.3787e-01,  5.5674e-01,  6.0395e-01,
          2.4957e-02, -6.9010e-02],
        [ 4.6783e-01,  9.5216e-03, -1.9368e-01,  4.7523e-01,  4.7549e-01,
         -1.0803e-01,  1.2652e-01],
        [ 1.2596e-02, -8.5964e-02,  6.2015e-02,  2.6655e-01, -7.2295e-02,
         -5.2671e-02, -6.9555e-02],
        [ 3.6165e-01,  3.4439e-01,  5.7684e-02, -2.6306e-01, -5.4794e-01,
          3.5743e-02, -2.9717e-01],
        [ 2.6537e-01, -8.4985e-01, -3.7734e-01,  1.2683e-01,  3.1132e-01,
          5.8659e-02,  2.3575e-01],
        [ 2.1492e-01,  3.9598e-01, -3.9039e-01, -5.2518e-02,  3.1711e-01,
          1.9138e-01,  3.7754e-01],
        [ 1.3581e-01,  3.7635e-01,  1.5227e-02,  4.4258e-01,  1.0014e-01,
         -5.1374e-02, -3.8747e-01],
        [ 1.6551e-01,  4.6681e-01,  2.7935e-01, -2.4598e-01,  4.3974e-02,
         -3.4024e